# 11. NMS와 탐지 평가

이 노트북은 `10_Bounding_Box와_IoU.ipynb` 다음 단계로, 객체 탐지 모델의 예측 결과를 정리하고 평가하는 방법을 다룹니다.

탐지 모델은 같은 객체 주변에 여러 개의 비슷한 박스를 출력할 수 있습니다. 따라서 높은 점수의 박스는 남기고, 같은 객체를 가리키는 중복 박스는 제거해야 합니다. 이 후처리 과정이 **NMS(Non-Maximum Suppression)** 입니다.

이번 노트북의 목표는 다음과 같습니다.

- NMS가 왜 필요한지 이해합니다.
- confidence score와 IoU threshold를 함께 사용하는 흐름을 익힙니다.
- TP, FP, FN의 의미를 객체 탐지 관점에서 정리합니다.
- precision, recall, AP의 기본 아이디어를 작은 예제로 확인합니다.


In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.unicode_minus'] = False


## 11-1. 기본 함수 준비

NMS와 평가 지표는 모두 박스 사이의 IoU를 기반으로 합니다. 이전 노트북에서 사용한 면적과 IoU 계산 함수를 다시 준비합니다.


In [ ]:
def box_area_xyxy(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_area = box_area_xyxy((inter_x1, inter_y1, inter_x2, inter_y2))

    union_area = box_area_xyxy(box_a) + box_area_xyxy(box_b) - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def draw_box(ax, box, label, color, linewidth=2.2):
    x1, y1, x2, y2 = box
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=linewidth))
    ax.text(x1, max(10, y1 - 4), label, color=color, fontsize=10, weight='bold')


def draw_predictions(predictions, kept_indices=None, title=''):
    kept_indices = set(kept_indices or [])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.set_xlim(0, 240)
    ax.set_ylim(180, 0)
    ax.set_facecolor('#f8fafc')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

    for idx, pred in enumerate(predictions):
        color = 'crimson' if idx in kept_indices else 'royalblue'
        label = f"P{idx}: {pred['score']:.2f}"
        draw_box(ax, pred['box'], label, color, linewidth=3 if idx in kept_indices else 1.5)
    plt.show()


## 11-2. 왜 NMS가 필요한가?

모델은 한 객체 주변에서 여러 후보 박스를 만들 수 있습니다. 사람 눈에는 모두 같은 고양이를 가리키는 박스처럼 보이지만, 모델 출력으로는 여러 개의 탐지가 됩니다.

이 상태를 그대로 사용하면 객체 하나를 여러 번 찾았다고 세게 됩니다. 그래서 가장 점수가 높은 박스를 대표로 남기고, 그 박스와 IoU가 큰 다른 박스들을 제거합니다.


In [ ]:
predictions = [
    {'box': (45, 35, 145, 125), 'score': 0.96, 'class': 'cat'},
    {'box': (50, 40, 148, 128), 'score': 0.89, 'class': 'cat'},
    {'box': (58, 48, 154, 132), 'score': 0.77, 'class': 'cat'},
    {'box': (150, 30, 220, 95), 'score': 0.68, 'class': 'cat'},
]

draw_predictions(predictions, title='NMS 적용 전: 중복 후보 박스')

for i in range(len(predictions)):
    for j in range(i + 1, len(predictions)):
        print(f"P{i} vs P{j}: IoU={iou_xyxy(predictions[i]['box'], predictions[j]['box']):.3f}")


## 11-3. NMS 알고리즘

NMS의 기본 절차는 단순합니다.

1. confidence score가 높은 순서대로 예측 박스를 정렬합니다.
2. 가장 점수가 높은 박스를 선택합니다.
3. 선택한 박스와 IoU가 큰 박스들을 제거합니다.
4. 남은 박스가 없을 때까지 반복합니다.

여기서는 같은 클래스 안에서만 NMS를 적용합니다. 실제 탐지 모델도 보통 클래스별로 NMS를 수행합니다.


In [ ]:
def nms(predictions, iou_threshold=0.5):
    order = sorted(range(len(predictions)), key=lambda idx: predictions[idx]['score'], reverse=True)
    keep = []

    while order:
        current = order.pop(0)
        keep.append(current)

        remaining = []
        for idx in order:
            iou = iou_xyxy(predictions[current]['box'], predictions[idx]['box'])
            if iou < iou_threshold:
                remaining.append(idx)
        order = remaining

    return keep


kept = nms(predictions, iou_threshold=0.5)
print('kept indices:', kept)
draw_predictions(predictions, kept_indices=kept, title='NMS 적용 후: 대표 박스만 유지')


## 11-4. threshold가 결과를 바꾼다

NMS의 IoU threshold가 낮으면 박스를 더 공격적으로 제거합니다. threshold가 높으면 많이 겹쳐도 남깁니다.

- 낮은 threshold: 중복 제거가 강함, 실제 다른 객체까지 지울 위험이 있음
- 높은 threshold: 중복 박스가 많이 남을 수 있음

탐지 모델을 사용할 때 threshold는 단순한 상수가 아니라 결과 품질을 조절하는 중요한 하이퍼파라미터입니다.


In [ ]:
for threshold in [0.3, 0.5, 0.8]:
    kept = nms(predictions, iou_threshold=threshold)
    kept_scores = [predictions[idx]['score'] for idx in kept]
    print(f'IoU threshold={threshold}: kept={kept}, scores={kept_scores}')


## 11-5. TP, FP, FN

객체 탐지 평가에서는 예측 박스와 정답 박스를 매칭한 뒤 결과를 다음처럼 나눕니다.

- TP(True Positive): 클래스가 맞고 IoU 기준도 통과한 예측
- FP(False Positive): 틀린 클래스이거나, IoU가 낮거나, 이미 매칭된 객체를 또 탐지한 예측
- FN(False Negative): 정답 객체가 있었지만 탐지하지 못한 경우

중요한 점은 하나의 정답 객체는 보통 하나의 예측과만 매칭된다는 것입니다. 같은 객체를 두 번 탐지하면 첫 번째는 TP가 될 수 있지만 나머지는 FP가 됩니다.


In [ ]:
ground_truths = [
    {'box': (45, 35, 145, 125), 'class': 'cat', 'matched': False},
    {'box': (150, 35, 220, 105), 'class': 'cat', 'matched': False},
]

eval_predictions = [
    {'box': (47, 37, 144, 124), 'class': 'cat', 'score': 0.95},
    {'box': (52, 42, 150, 130), 'class': 'cat', 'score': 0.86},
    {'box': (148, 34, 218, 108), 'class': 'cat', 'score': 0.80},
    {'box': (20, 120, 80, 170), 'class': 'cat', 'score': 0.62},
]

def match_predictions(predictions, ground_truths, iou_threshold=0.5):
    gt_state = [dict(gt, matched=False) for gt in ground_truths]
    results = []

    sorted_preds = sorted(predictions, key=lambda pred: pred['score'], reverse=True)
    for pred in sorted_preds:
        best_iou = 0.0
        best_gt_idx = None
        for gt_idx, gt in enumerate(gt_state):
            if gt['matched'] or pred['class'] != gt['class']:
                continue
            iou = iou_xyxy(pred['box'], gt['box'])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_gt_idx is not None and best_iou >= iou_threshold:
            gt_state[best_gt_idx]['matched'] = True
            results.append((pred, 'TP', best_iou))
        else:
            results.append((pred, 'FP', best_iou))

    fn = sum(not gt['matched'] for gt in gt_state)
    return results, fn


matched_results, fn = match_predictions(eval_predictions, ground_truths)
for pred, label, iou in matched_results:
    print(f"score={pred['score']:.2f}, IoU={iou:.3f} -> {label}")
print('FN:', fn)


## 11-6. Precision과 Recall

TP, FP, FN을 알면 precision과 recall을 계산할 수 있습니다.

$$Precision = \frac{TP}{TP + FP}$$

$$Recall = \frac{TP}{TP + FN}$$

- precision: 모델이 찾았다고 말한 것 중 실제로 맞은 비율
- recall: 실제 객체 중 모델이 찾아낸 비율

confidence threshold를 높이면 보통 precision은 좋아지고 recall은 낮아질 수 있습니다. 반대로 threshold를 낮추면 더 많이 찾지만 FP도 늘어날 수 있습니다.


In [ ]:
tp = sum(label == 'TP' for _, label, _ in matched_results)
fp = sum(label == 'FP' for _, label, _ in matched_results)

precision = tp / (tp + fp) if tp + fp > 0 else 0.0
recall = tp / (tp + fn) if tp + fn > 0 else 0.0

print('TP:', tp)
print('FP:', fp)
print('FN:', fn)
print('Precision:', round(precision, 3))
print('Recall   :', round(recall, 3))


## 11-7. Precision-Recall 곡선과 AP

AP(Average Precision)는 confidence threshold를 하나로 고정하지 않고, 점수 순서대로 예측을 하나씩 받아들이면서 precision과 recall의 변화를 요약한 값입니다.

아래 코드는 작은 예제에서 PR 곡선을 만들고, 간단한 면적 계산으로 AP의 직관을 확인합니다. 실제 COCO 평가의 mAP는 여러 IoU threshold와 클래스 평균을 사용하므로 더 엄밀합니다.


In [ ]:
labels_by_score = [label for _, label, _ in matched_results]
num_gt = len(ground_truths)
cum_tp = 0
cum_fp = 0
precisions = []
recalls = []

for label in labels_by_score:
    if label == 'TP':
        cum_tp += 1
    else:
        cum_fp += 1
    precisions.append(cum_tp / (cum_tp + cum_fp))
    recalls.append(cum_tp / num_gt)

ap = 0.0
prev_recall = 0.0
for precision_value, recall_value in zip(precisions, recalls):
    ap += precision_value * max(0, recall_value - prev_recall)
    prev_recall = recall_value

plt.figure(figsize=(6, 4))
plt.step(recalls, precisions, where='post', marker='o')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'간단한 PR 곡선 | AP={ap:.3f}')
plt.xlim(0, 1.05)
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.show()

print('recalls   :', [round(v, 3) for v in recalls])
print('precisions:', [round(v, 3) for v in precisions])
print('AP        :', round(ap, 3))


## 정리

- NMS는 같은 객체를 가리키는 중복 예측 박스를 제거하는 후처리입니다.
- confidence score가 높은 박스를 먼저 선택하고, IoU가 큰 나머지 박스를 억제합니다.
- TP, FP, FN은 탐지 결과를 평가하기 위한 기본 단위입니다.
- precision은 예측의 정확성, recall은 실제 객체를 놓치지 않는 정도를 나타냅니다.
- AP는 confidence threshold 변화에 따른 precision-recall 관계를 요약합니다.

다음 노트북 `12_객체_탐지의_초기_흐름.ipynb`에서는 sliding window, region proposal, R-CNN 계열로 이어지는 초기 객체 탐지 접근을 살펴봅니다.
